# Giao trễ và review xấu trong từng seller: panel seller × tháng

Module 03 ghi nhận: đơn trễ có tỉ lệ review xấu 54%, đơn đúng hẹn 9%. Nhưng seller kém có thể vừa
giao trễ vừa bán hàng kém, và tháng cao điểm (Black Friday 11/2017, đợt trễ 02–03/2018) vừa nhiều đơn
trễ vừa nhiều khách khó chịu. Panel seller × tháng với fixed effects tách được một phần các yếu tố đó.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` (dòng đầu `# explore`) đọc dữ liệu thật, vẽ, gọi hàm. Test bỏ qua chúng.
- Chấm: `uv run pytest tracks/data-science/modules/05-econometrics-regression/exercises/advanced/02-olist-seller-panel -q`
- Báo cáo viết ở `REPORT.md` cùng thư mục.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from linearmodels.panel import PanelOLS

In [ ]:
# explore: đọc Olist thật (tải bằng: uv run python data/download_olist.py)
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
RAW = ROOT / "data/raw"
orders = pd.read_csv(RAW / "olist_orders_dataset.csv",
                     parse_dates=["order_purchase_timestamp", "order_delivered_customer_date", "order_estimated_delivery_date"])
reviews = pd.read_csv(RAW / "olist_order_reviews_dataset.csv", parse_dates=["review_answer_timestamp"])
items = pd.read_csv(RAW / "olist_order_items_dataset.csv")

## 1. Dựng panel

Mỗi dòng là một seller trong một tháng mua hàng. Đơn có nhiều seller bị loại (không gán được trễ /
review cho một seller). Ô quá ít đơn cho tỉ lệ rất nhiễu, nên chỉ giữ ô có ít nhất `min_orders` đơn
và seller có ít nhất `min_months` ô. Hãy nghĩ: các bộ lọc này chọn ra loại seller nào?

In [ ]:
def seller_month_panel(orders, reviews, items, start="2017-01-01", end="2018-09-01", min_orders=5, min_months=6):
    """Panel seller × tháng mua hàng.

    Đơn được tính: giao xong (order_status == "delivered" và có order_delivered_customer_date), có
    review (mỗi đơn giữ review có review_answer_timestamp mới nhất), chỉ có MỘT seller (theo items),
    order_purchase_timestamp trong [start, end).
    late = order_delivered_customer_date > order_estimated_delivery_date (so nguyên timestamp);
    bad = review_score <= 2. month = đầu tháng của order_purchase_timestamp (Timestamp).

    Giữ ô (seller, month) có ít nhất min_orders đơn, rồi giữ seller còn ít nhất min_months ô.
    Trả DataFrame cột seller_id, month, n_orders, late_rate, bad_rate; sắp theo seller_id, month;
    index 0..n-1.
    """
    # TODO: đơn giao xong + late; review mới nhất + bad; đơn một seller; lọc thời gian; month
    # TODO: groupby (seller_id, month); lọc min_orders rồi min_months
    raise NotImplementedError

In [ ]:
# explore: panel trên dữ liệu thật
panel = seller_month_panel(orders, reviews, items)
print(len(panel), panel.seller_id.nunique(), panel.month.nunique())
panel.describe()

In [ ]:
# explore: trung bình theo tháng (cú sốc chung) và biến thiên trong từng seller
import matplotlib.pyplot as plt

by_month = panel.groupby("month")[["late_rate", "bad_rate"]].mean()
by_month.plot(marker="o", figsize=(10, 4), title="Trung bình theo tháng trên các ô seller × tháng")
plt.show()
within = panel.late_rate - panel.groupby("seller_id").late_rate.transform("mean")
print("độ lệch chuẩn late_rate: tổng", round(panel.late_rate.std(), 3), "| trong seller", round(within.std(), 3))

## 2. Ba spec

`bad_rate ~ late_rate`: pooled, seller FE, two-way FE (seller + tháng). SE cluster theo seller. Hệ số
thay đổi thế nào qua các spec, và mỗi bước loại bỏ loại confounder nào?

In [ ]:
def fit_specs(panel: pd.DataFrame) -> pd.DataFrame:
    """Hồi quy bad_rate ~ hằng số + late_rate, không trọng số, SE cluster theo seller (linearmodels).

    Ba spec: pooled (PanelOLS không effect), seller_fe (entity_effects), twfe (entity + time effects).
    Trả DataFrame index ["pooled", "seller_fe", "twfe"], cột ["coef", "se"] của late_rate.
    """
    # TODO: panel.set_index(["seller_id", "month"]); sm.add_constant(...[["late_rate"]])
    # TODO: PanelOLS(...).fit(cov_type="clustered", cluster_entity=True) cho từng spec
    raise NotImplementedError

In [ ]:
# explore: kết quả ba spec
fit_specs(panel).round(4)

## 3. Phân tích thêm (tự do)

Gợi ý: trọng số theo `n_orders`; đổi `min_orders` (5, 10, 20) và xem hệ số thay đổi thế nào (sai số đo
lường làm hệ số FE bị kéo về 0); biến trễ một tháng của `late_rate`; tác động theo nhóm seller. Viết
kết quả vào `REPORT.md`.

In [ ]:
# explore: chỗ cho phân tích của bạn